# Weekly Internship Report - Week 2
**Intern:** Mahi Savani (24AIML059)

## Task 7: Insights & Trends Analysis on Scraped Data

This notebook loads `scraped_books.csv` (produced in **Task 6**) and identifies key patterns, trends, and outliers, presenting the findings through charts.

> Run the Task 6 notebook first so that `scraped_books.csv` exists in the same working directory.

In [ ]:
# Install required libraries (uncomment if not already installed)
# !pip install pandas matplotlib seaborn


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
# Load the scraped dataset
df = pd.read_csv("scraped_books.csv")
print("Shape:", df.shape)
df.info()


In [ ]:
df.describe(include="all")


### Insight 1: Which categories have the most books listed?

In [ ]:
top_categories = df["category"].value_counts().head(10)

plt.figure()
sns.barplot(x=top_categories.values, y=top_categories.index, palette="viridis")
plt.title("Top 10 Book Categories by Count")
plt.xlabel("Number of Books")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig("top_categories.png")
plt.show()


### Insight 2: Price distribution and outliers

In [ ]:
plt.figure()
sns.histplot(df["price_gbp"], bins=20, kde=True, color="teal")
plt.title("Distribution of Book Prices (GBP)")
plt.xlabel("Price (£)")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.savefig("price_distribution.png")
plt.show()

plt.figure()
sns.boxplot(x=df["price_gbp"], color="skyblue")
plt.title("Boxplot of Book Prices (GBP)")
plt.xlabel("Price (£)")
plt.tight_layout()
plt.savefig("price_boxplot.png")
plt.show()

# Flag price outliers using the IQR method
Q1, Q3 = df["price_gbp"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = df[(df["price_gbp"] < lower) | (df["price_gbp"] > upper)]
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")
print(f"Number of price outliers: {len(outliers)}")
outliers[["title", "price_gbp", "category"]].head(10)


### Insight 3: Star rating trends

In [ ]:
rating_counts = df["rating"].value_counts().sort_index()

plt.figure()
sns.barplot(x=rating_counts.index, y=rating_counts.values, palette="magma")
plt.title("Distribution of Star Ratings")
plt.xlabel("Star Rating")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.savefig("rating_distribution.png")
plt.show()

print("Average rating:", round(df["rating"].mean(), 2))


### Insight 4: Does price vary with rating?

In [ ]:
plt.figure()
sns.boxplot(x="rating", y="price_gbp", data=df, palette="coolwarm")
plt.title("Price Spread by Star Rating")
plt.xlabel("Star Rating")
plt.ylabel("Price (£)")
plt.tight_layout()
plt.savefig("price_by_rating.png")
plt.show()

avg_price_by_rating = df.groupby("rating")["price_gbp"].mean().round(2)
print("Average price by rating:")
print(avg_price_by_rating)


### Insight 5: Stock availability

In [ ]:
stock_counts = df["in_stock"].value_counts()

plt.figure()
plt.pie(stock_counts.values, labels=stock_counts.index.map({True: "In Stock", False: "Out of Stock"}),
        autopct="%1.1f%%", colors=["#4CAF50", "#F44336"], startangle=90)
plt.title("In Stock vs Out of Stock")
plt.tight_layout()
plt.savefig("stock_availability.png")
plt.show()


### Insight 6: Average price by category (top 10 most & least expensive)

In [ ]:
category_price = df.groupby("category")["price_gbp"].agg(["mean", "count"]).round(2)
category_price = category_price[category_price["count"] >= 3]  # ignore tiny categories

top_expensive = category_price.sort_values("mean", ascending=False).head(10)
top_cheap = category_price.sort_values("mean", ascending=True).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=top_expensive["mean"], y=top_expensive.index, palette="rocket", ax=axes[0])
axes[0].set_title("Top 10 Most Expensive Categories (avg price)")
axes[0].set_xlabel("Average Price (£)")

sns.barplot(x=top_cheap["mean"], y=top_cheap.index, palette="crest", ax=axes[1])
axes[1].set_title("Top 10 Cheapest Categories (avg price)")
axes[1].set_xlabel("Average Price (£)")

plt.tight_layout()
plt.savefig("category_price_extremes.png")
plt.show()


### Insight 7: Price variability within each category (spread, not just average)

In [ ]:
category_spread = df.groupby("category")["price_gbp"].agg(["mean", "std", "min", "max", "count"]).round(2)
category_spread = category_spread[category_spread["count"] >= 3]
category_spread["range"] = category_spread["max"] - category_spread["min"]
most_variable = category_spread.sort_values("std", ascending=False).head(10)

plt.figure()
sns.barplot(x=most_variable["std"], y=most_variable.index, palette="flare")
plt.title("Top 10 Categories with Highest Price Variability (std dev)")
plt.xlabel("Price Std Dev (£)")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig("category_price_variability.png")
plt.show()

most_variable


### Insight 8: Top 10 most expensive & cheapest individual books

In [ ]:
top10_expensive = df.nlargest(10, "price_gbp")[["title", "price_gbp", "rating", "category"]]
top10_cheap = df.nsmallest(10, "price_gbp")[["title", "price_gbp", "rating", "category"]]

print("Most expensive books:")
display(top10_expensive)

print("\nCheapest books:")
display(top10_cheap)


### Insight 9: Correlation between price, rating, and stock status

In [ ]:
corr_df = df.copy()
corr_df["in_stock_num"] = corr_df["in_stock"].astype(int)
corr_matrix = corr_df[["price_gbp", "rating", "in_stock_num"]].corr().round(2)

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap: Price, Rating, Stock")
plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.show()

corr_matrix


### Insight 10: Category vs. average rating (which genres rate highest?)

In [ ]:
category_rating = df.groupby("category")["rating"].agg(["mean", "count"]).round(2)
category_rating = category_rating[category_rating["count"] >= 3]
top_rated_categories = category_rating.sort_values("mean", ascending=False).head(10)

plt.figure()
sns.barplot(x=top_rated_categories["mean"], y=top_rated_categories.index, palette="viridis")
plt.title("Top 10 Highest-Rated Categories (avg star rating)")
plt.xlabel("Average Rating")
plt.ylabel("Category")
plt.xlim(0, 5)
plt.tight_layout()
plt.savefig("category_avg_rating.png")
plt.show()


### Insight 11: Most common words in book titles (quick text trend check)

In [ ]:
from collections import Counter
import re as re_mod

stopwords = {"the", "a", "an", "of", "and", "to", "in", "for", "on", "with", "&", "-", "is", "at", "by"}
words = []
for title in df["title"]:
    tokens = re_mod.findall(r"[a-zA-Z]+", title.lower())
    words.extend([t for t in tokens if t not in stopwords and len(t) > 2])

word_freq = Counter(words).most_common(15)
word_df = pd.DataFrame(word_freq, columns=["word", "count"])

plt.figure()
sns.barplot(x=word_df["count"], y=word_df["word"], palette="mako")
plt.title("Top 15 Most Common Words in Book Titles")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.savefig("title_word_frequency.png")
plt.show()


### Insight 12: Are higher-priced books more/less likely to be in stock?

In [ ]:
plt.figure()
sns.boxplot(x="in_stock", y="price_gbp", data=df, palette="Set2")
plt.title("Price Distribution: In Stock vs Out of Stock")
plt.xlabel("In Stock")
plt.ylabel("Price (£)")
plt.tight_layout()
plt.savefig("price_by_stock.png")
plt.show()


### Summary of Findings
- **Category concentration**: A handful of categories account for a disproportionate share of the catalogue (Insight 1), indicating where inventory is concentrated.
- **Price patterns**: Prices are right-skewed with IQR-flagged outliers (Insight 2); certain categories run consistently more expensive or cheaper on average (Insight 6), and some categories show far more price variability than others (Insight 7).
- **Extremes**: The most expensive and cheapest individual books highlight pricing extremes worth spot-checking (Insight 8).
- **Correlations**: Price, rating, and stock status show only weak correlation with each other (Insight 9) — rating and price appear largely independent, meaning higher price does not reliably signal a better-rated book.
- **Ratings by genre**: Certain categories skew toward higher average ratings than others (Insight 10).
- **Title text trends**: Common recurring words in titles hint at popular themes/genres in the catalogue (Insight 11).
- **Stock vs price**: Stock status shows little to no systematic price difference (Insight 12), suggesting availability isn't price-driven here.

These steps mirror a typical exploratory data analysis (EDA) workflow: load → profile → visualize → correlate → flag outliers → summarize.